In [ ]:
#export 

import math
import logging
import subprocess
from pathlib import Path

tag="AUDIO_MANAGER"
"""
El módulo `AUDIO_MANAGER` es un **gestor de transcripción de audio** que proporciona una pipeline completa para procesar archivos de audio: 
- desde la limpieza y división 
- hasta la transcripción automática con reconocimiento de hablantes (diarización). 
    
Está diseñado para funcionar en entornos macOS usando modelos locales (mlx-community) y herramientas de FFmpeg.
    
El módulo genera una separación de responsabilidad generando tres clases distintas:
- AudioCleaner para realizar la limpieza de audio, usando herramientas diversas. 
- SpeechTranscriber para realizar las transcripciones usando motores distintos.
- AudioTranscription para realizar el pipeline completo. 
    
# 1. `AudioCleaner` — Limpieza de audio
**Responsabilidad:** Aplicar filtros de procesamiento de audio para mejorar la calidad del audio antes de la transcripción.

- **`clean(input_file, method, output_file)`**: Método principal que delega según el método elegido.
- **`_clean_ffmpeg()`**: Aplica filtros FFmpeg (highpass/lowpass, normalización de volumen, loudnorm) y convierte a mono 16kHz.
- **`_clean_deepfilter()`**: Convierte el audio a WAV mono 48kHz, ejecuta DeepFilterNet para reducción de ruido, y limpia archivos temporales.
- **`_run_command()`**: Ejecuta comandos de sistema y maneja errores.
- **`_safe_unlink()`**: Elimina archivos de forma segura.

**Métodos de limpieza disponibles:**
`CLEAN_NONE`        --> Sin limpieza
`CLEAN_FFMPEG`      --> Filtros FFmpeg (highpass/lowpass/loudnorm)
`CLEAN_DEEPFILTER`  --> DeepFilterNet para reducción de ruido avanzada

# 2. `SpeechTranscriber` — Transcripción de voz a texto
**Responsabilidad:** Transcribir archivos de audio a texto usando modelos de ASR (Automatic Speech Recognition) locales.

- **`transcribe(audio_file)`**: Método principal que transcribe y opcionalmente aplica diarización.
- **`_transcribe_whisper()`**: Usa `mlx-whisper` para transcripción.
- **`_transcribe_mlx_audio()`**: Usa `mlx-audio` STT como alternativa (prueba dos APIs diferentes).
- **`_diarize()`**: Realiza diarización (identificación de hablantes) usando `mlx-audio VAD`.
- **`_normalize_asr_segments()`**: Normaliza diferentes formatos de salida de ASR a un formato estándar de segmentos.
- **`_plain_text()`**: Devuelve el texto plano sin diarización.
- **`_merge_asr_with_diarization()`**: Fusiona los segmentos de texto con los hablantes identificados.

**Engines disponibles:**
`ENGINE_WHISPER`    -->  Modelo Whisper (mlx-community)
`ENGINE_MLX_AUDIO`  --> Modelo mlx-audio STT

# 3. `AudioTranscription` — Pipeline completa
**Responsabilidad:** Orquestar el flujo completo de procesamiento de audio: división, limpieza, transcripción y generación de archivos de salida.

**Parámetros de configuración:**
`engine`                --> Motor de transcripción (`whisper` o `mlx-audio`) (por defecto whisper)
`diarization`           --> Activar/desactivar reconocimiento de hablantes (por defecto si)
`clean_method`          --> Método de limpieza de audio (por defecto no)
`split`                 --> Dividir el audio en segmentos más cortos (por defecto no)
`segment_minutes`       --> Duración en minutos de cada segmento (por defecto 25)
`output_dir`            --> Directorio de salida (por defecto procesados en el directorio origen)
`whisper_model`         --> Modelo Whisper a usar (por defecto grande)
`mlx_audio_asr_model`   --> Modelo mlx-audio a usar (por defecto xxxx)
`diar_model`            --> Modelo de diarización
`diar_threshold`        --> Umbral para diarización

**Métodos principales:**
- **`process_directory(input dir)`**: Ejecuta la pipeline completa para un directorio donde hay varios file.
- **`process(input_file)`**: Ejecuta la pipeline completa para un archivo de audio.
  1. Divide el audio si `split=True`
  2. Limpia cada segmento si se configura
  3. Transcribe cada segmento
  4. Genera archivos `.txt` por segmento
  5. Genera un archivo de transcripción final combinado

- **`divideFichero(input_file, minutos, output_dir)`**: Divide un archivo de audio en segmentos de duración configurable.
- **`_write_final_txt(txt_segmentos, output_file)`**: Combina todas las transcripciones parciales en un único archivo final.
- **`get_audio_duration(input_file)`**: Obtiene la duración de un archivo de audio usando `ffprobe`.

## Funciones de utilidad
- **`format_min_sec(seconds)`**: Convierta segundos a formato `MM:SS`.
- **`format_min_sec_filename(seconds)`**: Convierta segundos a formato para nombres de archivo `MMmSSs`.

## Modo de uso

### Uso básico con `AudioTranscription`:

```python
from transcribe_manager import AudioTranscription

# Crear pipeline con configuración por defecto
pipeline = AudioTranscription(
    engine=AudioTranscription.ENGINE_WHISPER,
    diarization=True,
    clean_method=AudioTranscriptionPipeline.CLEAN_FFMPEG,
    split=True,
    segment_minutes=25,
    output_dir="./output"
)

# Procesar un archivo de audio
output_file = pipeline.process("grabacion.mp3")
# Devuelve la ruta al archivo .txt con la transcripción final
```

### Uso directo de `AudioCleaner`:
```python
from transcribe_manager import AudioCleaner

cleaner = AudioCleaner()
resultado = cleaner.clean(
    input_file="audio_original.mp3",
    method=AudioCleaner.CLEAN_FFMPEG,
    output_file="audio_limpio.wav"
)
```

### Uso directo de `SpeechTranscriber`:
```python
from transcribe_manager import SpeechTranscriber

transcriber = SpeechTranscriber(
    engine=SpeechTranscriber.ENGINE_WHISPER,
    diarization=True
)
texto = transcriber.transcribe("audio_limpio.wav")
print(texto)
```

## Flujo de trabajo de la pipeline
Archivo de entrada
       │
       ▼
  [Opcional] Dividir en segmentos (FFmpeg)
       │
       ▼
  [Opcional] Limpiar audio (FFmpeg / DeepFilterNet)
       │
       ▼
  Transcribir (Whisper / mlx-audio)
       │
       ▼
  [Opcional] Aplicar diarización (identificar hablantes)
       │
       ▼
  Generar .txt por segmento
       │
       ▼
  Combinar en transcripción_final.txt

## Dependencias externas

- **FFmpeg / FFprobe**: Procesamiento y división de audio
- **mlx-whisper**: Modelo de transcripción Whisper
- **mlx-audio**: Modelo ASR alternativo + diarización (VAD)
- **DeepFilter**: Red neuronal para reducción de ruido (opcional)
"""


'\nEl módulo `AUDIO_MANAGER` es un **gestor de transcripción de audio** que proporciona una pipeline completa para procesar archivos de audio: \n- desde la limpieza y división \n- hasta la transcripción automática con reconocimiento de hablantes (diarización). \n\nEstá diseñado para funcionar en entornos macOS usando modelos locales (mlx-community) y herramientas de FFmpeg.\n\nEl módulo genera una separación de responsabilidad generando tres clases distintas:\n- AudioCleaner para realizar la limpieza de audio, usando herramientas diversas. \n- SpeechTranscriber para realizar las transcripciones usando motores distintos.\n- AudioTranscription para realizar el pipeline completo. \n\n# 1. `AudioCleaner` — Limpieza de audio\n**Responsabilidad:** Aplicar filtros de procesamiento de audio para mejorar la calidad del audio antes de la transcripción.\n\n- **`clean(input_file, method, output_file)`**: Método principal que delega según el método elegido.\n- **`_clean_ffmpeg()`**: Aplica filtros 

In [ ]:
#export

class AudioCleaner:
    """
    Gestor de limpieza y procesamiento de archivos de audio.
    
    Proporciona métodos para aplicar filtros y procesamiento de audio
    utilizando diferentes estrategias (FFmpeg, DeepFilterNet) para mejorar
    la calidad del audio antes de la transcripción.
    
    Atributos de clase:
        CLEAN_NONE (str): Identificador para no aplicar limpieza.
        CLEAN_FFMPEG (str): Identificador para limpieza con FFmpeg.
        CLEAN_DEEPFILTER (str): Identificador para limpieza con DeepFilterNet.
    
    Ejemplo:
        >>> cleaner = AudioCleaner()
        >>> output = cleaner.clean(
        ...     input_file="audio.mp3",
        ...     method=AudioCleaner.CLEAN_FFMPEG,
        ...     output_file="audio_limpio.wav"
        ... )
    """
    
    CLEAN_NONE = "none"
    CLEAN_FFMPEG = "ffmpeg"
    CLEAN_DEEPFILTER = "deepfilter"
    
    def __init__(self):
        """Inicializa el limpiador de audio con su logger."""
        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.debug("AudioCleaner inicializado")

    def clean(self, input_file, method=CLEAN_NONE, output_file=None):
        """
        Limpia un archivo de audio usando el método especificado.
        
        Aplica el método de limpieza seleccionado al archivo de audio de entrada.
        Si no se especifica archivo de salida, genera automáticamente un nombre
        basado en el archivo de entrada.
        
        Args:
            input_file (str or Path): Ruta del archivo de audio a limpiar.
            method (str, optional): Método de limpieza a aplicar.
                Valores válidos: CLEAN_NONE, CLEAN_FFMPEG, CLEAN_DEEPFILTER.
                Por defecto: CLEAN_NONE.
            output_file (str or Path, optional): Ruta del archivo de salida.
                Si es None, se genera automáticamente. Por defecto: None.
        
        Returns:
            str: Ruta del archivo de audio limpio (convertido a string).
        
        Raises:
            ValueError: Si el método especificado no es válido.
        
        Ejemplo:
            >>> cleaner = AudioCleaner()
            >>> limpio = cleaner.clean("entrada.mp3", AudioCleaner.CLEAN_FFMPEG)
        """
        self.logger.info(f"Limpieza de audio iniciada: {input_file}, método: {method}")
        
        if method == self.CLEAN_NONE:
            self.logger.debug(f"Método CLEAN_NONE: devolviendo archivo sin procesar")
            return str(input_file)

        if method == self.CLEAN_FFMPEG:
            self.logger.info(f"Aplicando limpieza FFmpeg a: {input_file}")
            return self.clean_ffmpeg(input_file, output_file)

        if method == self.CLEAN_DEEPFILTER:
            self.logger.info(f"Aplicando limpieza DeepFilterNet a: {input_file}")
            return self.clean_deepfilter(input_file, output_file)

        error_msg = (
            f"Método de limpieza no soportado: {method}. "
            f"Usa AudioCleaner.CLEAN_NONE, AudioCleaner.CLEAN_FFMPEG o AudioCleaner.CLEAN_DEEPFILTER."
        )
        self.logger.error(error_msg)
        raise ValueError(error_msg)

    def clean_ffmpeg(self, input_file, output_file=None):
        """
        Limpia audio aplicando filtros FFmpeg.
        
        Aplica los siguientes filtros de audio:
        - Filtro paso-alto (highpass, 100 Hz): Elimina ruido grave
        - Filtro paso-bajo (lowpass, 5500 Hz): Elimina ruido agudo
        - Aumento de volumen (8 dB)
        - Normalización de volumen (loudnorm LUFS)
        
        Convierte el resultado a:
        - Formato: WAV PCM 16-bit
        - Canales: Mono
        - Frecuencia de muestreo: 16 kHz
        
        Args:
            input_file (str or Path): Ruta del archivo de audio a limpiar.
            output_file (str or Path, optional): Ruta del archivo de salida.
                Si es None, genera automáticamente el nombre basado en
                el archivo de entrada con sufijo "_limpia.wav".
                Por defecto: None.
        
        Returns:
            str: Ruta del archivo de audio limpio (convertido a string).
        
        Raises:
            RuntimeError: Si FFmpeg falla durante la ejecución.
        
        Ejemplo:
            >>> cleaner = AudioCleaner()
            >>> salida = cleaner.clean_ffmpeg("entrada.mp3", "salida.wav")
        """
        self.logger.debug(f"clean_ffmpeg iniciado: {input_file}")
        input_file = Path(input_file)

        if output_file is None:
            output_file = input_file.parent / f"{input_file.stem}_limpia.wav"
            self.logger.debug(f"Nombre de salida generado automáticamente: {output_file}")
        else:
            output_file = Path(output_file)

        filtro_audio = (
            "highpass=f=100,"
            "lowpass=f=5500,"
            "volume=8dB,"
            "loudnorm=I=-16:LRA=11:TP=-1.5"
        )
        self.logger.debug(f"Filtros FFmpeg: {filtro_audio}")

        comando = [
            "ffmpeg",
            "-i", str(input_file),
            "-vn",
            "-af", filtro_audio,
            "-ac", "1",
            "-ar", "16000",
            "-c:a", "pcm_s16le",
            "-y",
            str(output_file),
        ]

        self.logger.debug(f"Ejecutando comando FFmpeg: {' '.join(comando)}")
        self._run_command(comando, "ERROR limpiando audio con FFmpeg:")
        
        self.logger.info(f"Audio limpio con FFmpeg guardado en: {output_file}")
        return str(output_file)

    def clean_deepfilter(self, input_file, output_file=None):
        """
        Limpia audio usando DeepFilterNet para reducción avanzada de ruido.
        
        DeepFilterNet es una red neuronal profunda especializada en reducción
        de ruido. El proceso es:
        1. Convierte el audio a WAV mono a 48 kHz (formato óptimo para DeepFilter)
        2. Ejecuta DeepFilterNet para eliminar ruido
        3. Detecta automáticamente el archivo de salida generado por DeepFilter
        4. Limpia archivos temporales
        
        Args:
            input_file (str or Path): Ruta del archivo de audio a limpiar.
            output_file (str or Path, optional): Ruta del archivo de salida final.
                Si es None, genera automáticamente el nombre basado en
                el archivo de entrada con sufijo "_limpia.wav".
                Por defecto: None.
        
        Returns:
            str: Ruta del archivo de audio limpio (convertido a string).
        
        Raises:
            RuntimeError: Si falla la conversión de formato o ejecución de DeepFilter.
            FileNotFoundError: Si DeepFilter no genera el archivo de salida esperado.
        
        Ejemplo:
            >>> cleaner = AudioCleaner()
            >>> salida = cleaner.clean_deepfilter("entrada.mp3", "salida.wav")
        """
        self.logger.debug(f"clean_deepfilter iniciado: {input_file}")
        input_file = Path(input_file)

        if output_file is None:
            output_file = input_file.parent / f"{input_file.stem}_limpia.wav"
            self.logger.debug(f"Nombre de salida generado automáticamente: {output_file}")
        else:
            output_file = Path(output_file)

        temp_48k = input_file.parent / f"{input_file.stem}_tmp_48k.wav"
        deepfilter_out_dir = input_file.parent

        # Paso 1: Convertir a 48 kHz mono
        self.logger.info(f"Convirtiendo {input_file} a WAV mono 48 kHz")
        cmd_convert_48k = [
            "ffmpeg",
            "-i", str(input_file),
            "-vn",
            "-ac", "1",
            "-ar", "48000",
            "-c:a", "pcm_s16le",
            "-y",
            str(temp_48k),
        ]

        self.logger.debug(f"Comando de conversión: {' '.join(cmd_convert_48k)}")
        self._run_command(
            cmd_convert_48k,
            "ERROR convirtiendo audio a WAV mono 48 kHz:",
        )
        self.logger.debug(f"Conversión completada: {temp_48k}")

        # Paso 2: Ejecutar DeepFilter
        cmd_deepfilter = [
            "deepFilter",
            str(temp_48k),
            "-o",
            str(deepfilter_out_dir),
        ]

        try:
            self.logger.info(f"Ejecutando DeepFilterNet en: {temp_48k}")
            self.logger.debug(f"Comando DeepFilter: {' '.join(cmd_deepfilter)}")
            self._run_command(
                cmd_deepfilter,
                "ERROR ejecutando DeepFilterNet:",
            )

            # Paso 3: Localizar archivo de salida
            self.logger.debug(f"Buscando salida DeepFilterNet en: {deepfilter_out_dir}")
            posibles_salidas = sorted(
                deepfilter_out_dir.glob(f"{temp_48k.stem}*DeepFilterNet*.wav")
            )

            if not posibles_salidas:
                error_msg = (
                    f"No se encontró salida WAV de DeepFilterNet para {temp_48k.name}"
                )
                self.logger.error(error_msg)
                raise FileNotFoundError(error_msg)

            enhanced_48k = posibles_salidas[0]
            self.logger.debug(f"Archivo de salida DeepFilter encontrado: {enhanced_48k}")

            # Paso 4: Renombrar archivo final
            if output_file.exists():
                self.logger.debug(f"Eliminando archivo existente: {output_file}")
                output_file.unlink()

            enhanced_48k.rename(output_file)
            self.logger.info(f"Audio limpio con DeepFilterNet guardado en: {output_file}")

            return str(output_file)

        finally:
            self.logger.debug(f"Limpiando archivo temporal: {temp_48k}")
            self._safe_unlink(temp_48k)

    @staticmethod
    def _run_command(comando, error_message):
        """
        Ejecuta un comando de shell y valida su resultado.
        
        Ejecuta el comando especificado capturando stdout y stderr.
        Si el comando falla (retorna código de error diferente de 0),
        lanza una excepción con el mensaje de error y la salida de stderr.
        
        Args:
            comando (list): Lista con el comando y sus argumentos.
                Ejemplo: ["ffmpeg", "-i", "input.mp3", "output.wav"]
            error_message (str): Mensaje de error personalizado a mostrar
                en caso de que falle el comando.
        
        Returns:
            subprocess.CompletedProcess: Resultado de la ejecución del comando
                con stdout y stderr capturados.
        
        Raises:
            RuntimeError: Si el comando falla (retorna código no-cero).
                El mensaje de error incluye el comando ejecutado y
                la salida de stderr para diagnóstico.
        
        Ejemplo:
            >>> AudioCleaner._run_command(
            ...     ["ffmpeg", "-version"],
            ...     "Error obteniendo versión de FFmpeg"
            ... )
        """
        logger = logging.getLogger(AudioCleaner.__name__)
        logger.debug(f"Ejecutando comando: {' '.join(comando)}")
        
        resultado = subprocess.run(
            comando,
            capture_output=True,
            text=True,
        )

        if resultado.returncode != 0:
            error_msg = (
                f"{error_message}\n\n"
                f"Comando:\n{' '.join(comando)}\n\n"
                f"STDERR:\n{resultado.stderr}"
            )
            logger.error(error_msg)
            raise RuntimeError(error_msg)

        logger.debug(f"Comando ejecutado exitosamente: {comando[0]}")
        return resultado

    @staticmethod
    def _safe_unlink(file_path):
        """
        Elimina un archivo de forma segura, ignorando errores.
        
        Intenta eliminar el archivo especificado. Si falla por cualquier
        motivo (archivo no existe, permisos insuficientes, etc.), el error
        se captura y se ignora silenciosamente.
        
        Args:
            file_path (str or Path): Ruta del archivo a eliminar.
        
        Ejemplo:
            >>> AudioCleaner._safe_unlink("/tmp/archivo_temporal.wav")
            # No lanza excepción aunque el archivo no exista
        """
        logger = logging.getLogger(AudioCleaner.__name__)
        try:
            Path(file_path).unlink()
            logger.debug(f"Archivo eliminado: {file_path}")
        except Exception as e:
            logger.debug(f"No se pudo eliminar {file_path}: {type(e).__name__}")
            pass


In [ ]:
#export

class SpeechTranscriber:
    """
    Transcriptor de voz a texto con soporte para diarización (identificación de hablantes).
    
    Proporciona métodos para transcribir archivos de audio usando modelos de ASR
    (Automatic Speech Recognition) locales. Soporta múltiples engines de transcripción
    y opcionalmente puede identificar diferentes hablantes (diarización).
    
    Atributos de clase:
        ENGINE_WHISPER (str): Identificador para usar modelo Whisper (mlx-community).
        ENGINE_MLX_AUDIO (str): Identificador para usar modelo mlx-audio STT.
    
    Ejemplo:
        >>> transcriber = SpeechTranscriber(
        ...     engine=SpeechTranscriber.ENGINE_WHISPER,
        ...     diarization=True
        ... )
        >>> texto = transcriber.transcribe("audio.wav")
        >>> print(texto)
    """
    
    ENGINE_WHISPER = "whisper"
    ENGINE_MLX_AUDIO = "mlx-audio"

    def __init__(
        self,
        engine=ENGINE_WHISPER,
        diarization=True,
        whisper_model="mlx-community/whisper-large-v3-mlx",
        mlx_audio_asr_model="mlx-community/parakeet-tdt-0.6b-v2",
        diar_model="mlx-community/diar_sortformer_4spk-v1-fp32",
        diar_threshold=0.5,
    ):
        """
        Inicializa el transcriptor de voz a texto.
        
        Args:
            engine (str, optional): Motor de transcripción a usar.
                Valores válidos: ENGINE_WHISPER, ENGINE_MLX_AUDIO.
                Por defecto: ENGINE_WHISPER.
            diarization (bool, optional): Activar identificación de hablantes.
                Por defecto: True.
            whisper_model (str, optional): Ruta del modelo Whisper a descargar.
                Por defecto: "mlx-community/whisper-large-v3-mlx".
            mlx_audio_asr_model (str, optional): Ruta del modelo mlx-audio ASR.
                Por defecto: "mlx-community/parakeet-tdt-0.6b-v2".
            diar_model (str, optional): Ruta del modelo de diarización.
                Por defecto: "mlx-community/diar_sortformer_4spk-v1-fp32".
            diar_threshold (float, optional): Umbral de confianza para diarización
                (0.0 a 1.0). Por defecto: 0.5.
        
        Raises:
            ValueError: Si el engine especificado no es válido.
        """
        self.logger = logging.getLogger(f"{self.__class__.__name__}.__init__")
        
        valid_engines = {
            self.ENGINE_WHISPER,
            self.ENGINE_MLX_AUDIO,
        }

        if engine not in valid_engines:
            error_msg = (
                f"Engine no soportado: {engine}. "
                f"Usa SpeechTranscriber.ENGINE_WHISPER o SpeechTranscriber.ENGINE_MLX_AUDIO."
            )
            self.logger.error(error_msg)
            raise ValueError(error_msg)

        self.engine = engine
        self.diarization = diarization
        self.whisper_model = whisper_model
        self.mlx_audio_asr_model = mlx_audio_asr_model
        self.diar_model = diar_model
        self.diar_threshold = diar_threshold
        
        self.logger.info(
            f"SpeechTranscriber inicializado: engine={engine}, "
            f"diarization={diarization}, threshold={diar_threshold}"
        )

    def transcribe(self, audio_file):
        """
        Transcribe un archivo de audio a texto.
        
        Realiza la transcripción del archivo de audio usando el engine configurado.
        Opcionalmente, aplica diarización para identificar diferentes hablantes.
        
        Args:
            audio_file (str or Path): Ruta del archivo de audio a transcribir.
        
        Returns:
            str: Texto transcrito. Si diarización está activa, incluye timestamps
                y nombres de hablantes en formato:
                "[00:05 - 00:12] Speaker_0: texto transcrito..."
                Si diarización está desactiva, devuelve texto plano concatenado.
        
        Raises:
            ValueError: Si el engine no es válido.
            RuntimeError: Si falla la transcripción con el engine seleccionado.
        
        Ejemplo:
            >>> transcriber = SpeechTranscriber(diarization=True)
            >>> resultado = transcriber.transcribe("grabacion.wav")
        """
        logger = logging.getLogger(f"{self.__class__.__name__}.transcribe")
        logger.info(f"Iniciando transcripción de: {audio_file}")
        
        audio_file = Path(audio_file)

        if self.engine == self.ENGINE_WHISPER:
            logger.debug(f"Usando engine Whisper para transcribir")
            asr_result = self._transcribe_whisper(audio_file)
        elif self.engine == self.ENGINE_MLX_AUDIO:
            logger.debug(f"Usando engine mlx-audio para transcribir")
            asr_result = self._transcribe_mlx_audio(audio_file)
        else:
            error_msg = f"Engine no soportado: {self.engine}"
            logger.error(error_msg)
            raise ValueError(error_msg)

        logger.debug(f"Normalizando segmentos ASR")
        asr_segments = self._normalize_asr_segments(asr_result)

        if not self.diarization:
            logger.info(f"Diarización desactiva, devolviendo texto plano")
            return self._plain_text(asr_segments)

        logger.info(f"Aplicando diarización a {len(asr_segments)} segmentos")
        diar_result = self._diarize(audio_file)

        logger.info(f"Fusionando ASR con diarización")
        return self._merge_asr_with_diarization(
            asr_segments,
            diar_result,
        )

    def _transcribe_whisper(self, audio_file):
        """
        Transcribe usando el modelo Whisper de mlx-community.
        
        Args:
            audio_file (Path): Ruta del archivo de audio.
        
        Returns:
            dict: Resultado de Whisper con keys 'text' y 'segments'.
        
        Raises:
            RuntimeError: Si falla la importación o ejecución de mlx_whisper.
        """
        logger = logging.getLogger(f"{self.__class__.__name__}._transcribe_whisper")
        logger.info(f"Transcribiendo con Whisper: {audio_file}")
        logger.debug(f"Modelo: {self.whisper_model}")
        
        try:
            from mlx_whisper import transcribe
            
            result = transcribe(
                str(audio_file),
                path_or_hf_repo=self.whisper_model,
            )
            logger.info(f"Transcripción Whisper completada exitosamente")
            return result
        except Exception as e:
            logger.error(f"Error en transcripción Whisper: {str(e)}", exc_info=True)
            raise

    def _transcribe_mlx_audio(self, audio_file):
        """
        Transcribe usando el modelo mlx-audio STT.
        
        La API de mlx-audio puede variar entre versiones, por lo que se
        intentan dos formas habituales de acceso al modelo.
        
        Args:
            audio_file (Path): Ruta del archivo de audio.
        
        Returns:
            Resultado de mlx-audio STT (puede ser dict, str o similar).
        
        Raises:
            RuntimeError: Si falla la transcripción con mlx-audio.
        """
        logger = logging.getLogger(f"{self.__class__.__name__}._transcribe_mlx_audio")
        logger.info(f"Transcribiendo con mlx-audio: {audio_file}")
        logger.debug(f"Modelo: {self.mlx_audio_asr_model}")

        try:
            from mlx_audio.stt import load as load_stt

            logger.debug(f"Cargando modelo mlx-audio STT (método 1: load)")
            model = load_stt(self.mlx_audio_asr_model)

            logger.debug(f"Generando transcripción")
            result = model.generate(
                str(audio_file),
                verbose=False,
            )
            logger.info(f"Transcripción mlx-audio completada exitosamente")
            return result

        except Exception as e:
            logger.debug(f"Método 1 falló: {type(e).__name__}, intentando método 2")

        try:
            from mlx_audio.stt import transcribe

            logger.debug(f"Cargando modelo mlx-audio STT (método 2: transcribe)")
            result = transcribe(
                str(audio_file),
                model=self.mlx_audio_asr_model,
            )
            logger.info(f"Transcripción mlx-audio completada exitosamente (método 2)")
            return result

        except Exception as e:
            error_msg = (
                "No he podido transcribir con mlx-audio. "
                "Comprueba tu versión con: python -m pip install -U mlx-audio"
            )
            logger.error(error_msg, exc_info=True)
            raise RuntimeError(error_msg) from e

    def _diarize(self, audio_file):
        """
        Realiza diarización (identificación de hablantes) en el audio.
        
        Utiliza el modelo VAD (Voice Activity Detection) de mlx-audio para
        detectar diferentes hablantes y sus intervalos de tiempo en el audio.
        
        Args:
            audio_file (Path): Ruta del archivo de audio.
        
        Returns:
            Objeto con segmentos de diarización, cada uno con:
            - start (float): Tiempo de inicio en segundos
            - end (float): Tiempo de finalización en segundos
            - speaker (str): Identificador del hablante (ej: "Speaker_0")
        
        Raises:
            RuntimeError: Si falla la carga o ejecución del modelo de diarización.
        """
        logger = logging.getLogger(f"{self.__class__.__name__}._diarize")
        logger.info(f"Iniciando diarización: {audio_file}")
        logger.debug(f"Modelo: {self.diar_model}, umbral: {self.diar_threshold}")
        
        try:
            from mlx_audio.vad import load as load_vad

            logger.debug(f"Cargando modelo de diarización")
            diarizer = load_vad(self.diar_model)

            logger.debug(f"Generando segmentos de diarización")
            result = diarizer.generate(
                str(audio_file),
                threshold=self.diar_threshold,
                verbose=False,
            )
            logger.info(f"Diarización completada exitosamente")
            return result
        except Exception as e:
            logger.error(f"Error en diarización: {str(e)}", exc_info=True)
            raise

    def _normalize_asr_segments(self, asr_result):
        """
        Normaliza diferentes formatos de salida ASR a un formato estándar.
        
        Convierte la salida del engine ASR (que puede variar entre Whisper,
        mlx-audio y otras variantes) a un formato uniforme de segmentos:
        [
            {"start": float, "end": float, "text": str},
            ...
        ]
        
        Soporta múltiples formatos:
        - Dict con key "segments" (formato Whisper)
        - Dict con key "text" (salida simple)
        - Objeto con atributo "segments" (objeto personalizado)
        - Objeto con atributo "text" (objeto simple)
        - String plano (texto sin segmentar)
        
        Args:
            asr_result: Resultado del engine ASR (puede ser de varios tipos).
        
        Returns:
            list: Lista de dicts con estructura uniforme de segmentos.
        
        Raises:
            ValueError: Si el formato de asr_result no es reconocido.
        """
        logger = logging.getLogger(f"{self.__class__.__name__}._normalize_asr_segments")
        logger.debug(f"Normalizando segmentos ASR, tipo: {type(asr_result).__name__}")

        if isinstance(asr_result, dict):
            if "segments" in asr_result:
                logger.debug(f"Detectado formato dict con 'segments'")
                return asr_result["segments"]

            if "text" in asr_result:
                logger.debug(f"Detectado formato dict con 'text'")
                return [
                    {
                        "start": 0.0,
                        "end": 0.0,
                        "text": asr_result["text"],
                    }
                ]

        if hasattr(asr_result, "segments"):
            logger.debug(f"Detectado objeto con atributo 'segments'")
            segmentos = []

            for seg in asr_result.segments:
                if isinstance(seg, dict):
                    start = seg.get("start", 0.0)
                    end = seg.get("end", 0.0)
                    text = seg.get("text", "")
                else:
                    start = getattr(seg, "start", 0.0)
                    end = getattr(seg, "end", 0.0)
                    text = getattr(seg, "text", "")

                segmentos.append(
                    {
                        "start": start,
                        "end": end,
                        "text": text,
                    }
                )

            logger.debug(f"Normalizados {len(segmentos)} segmentos de objeto")
            return segmentos

        if hasattr(asr_result, "text"):
            logger.debug(f"Detectado objeto con atributo 'text'")
            return [
                {
                    "start": 0.0,
                    "end": 0.0,
                    "text": asr_result.text,
                }
            ]

        if isinstance(asr_result, str):
            logger.debug(f"Detectado string plano")
            return [
                {
                    "start": 0.0,
                    "end": 0.0,
                    "text": asr_result,
                }
            ]

        error_msg = f"No sé interpretar la salida ASR: {type(asr_result)}"
        logger.error(error_msg)
        raise ValueError(error_msg)

    def _plain_text(self, asr_segments):
        """
        Extrae texto plano de los segmentos ASR sin timestamps ni diarización.
        
        Concatena el texto de todos los segmentos con espacios,
        eliminando espacios en blanco adicionales.
        
        Args:
            asr_segments (list): Lista de dicts con estructura de segmentos.
        
        Returns:
            str: Texto concatenado de todos los segmentos.
        
        Ejemplo:
            >>> segmentos = [
            ...     {"start": 0, "end": 5, "text": "Hola"},
            ...     {"start": 5, "end": 10, "text": "mundo"}
            ... ]
            >>> text = transcriber._plain_text(segmentos)
            >>> print(text)
            'Hola mundo'
        """
        logger = logging.getLogger(f"{self.__class__.__name__}._plain_text")
        logger.debug(f"Extrayendo texto plano de {len(asr_segments)} segmentos")
        
        textos = []

        for seg in asr_segments:
            text = str(seg.get("text", "")).strip()

            if text:
                textos.append(text)

        resultado = " ".join(textos)
        logger.debug(f"Texto plano extraído: {len(resultado)} caracteres")
        return resultado

    def _merge_asr_with_diarization(self, asr_segments, diar_result):
        """
        Fusiona transcripción ASR con identificación de hablantes (diarización).
        
        Combina los segmentos de texto transcrito con la información de diarización
        para generar un formato de salida que incluya timestamps, hablante e texto:
        
        Formato de salida:
        [00:05 - 00:12] Speaker_0: texto del primer hablante...
        [00:12 - 00:18] Speaker_1: texto del segundo hablante...
        
        Algoritmo:
        - Para cada segmento ASR, calcula el máximo solapamiento temporal
          con los segmentos de diarización para determinar el hablante.
        - Si no hay solapamiento, asigna hablante "unknown".
        - Ignora segmentos ASR vacíos.
        
        Args:
            asr_segments (list): Lista de dicts con estructura ASR estándar.
            diar_result: Objeto con atributo "segments" conteniendo hablantes.
        
        Returns:
            str: Texto formateado con timestamps y hablantes, separado por newlines.
        
        Ejemplo:
            >>> output = transcriber._merge_asr_with_diarization(
            ...     asr_segments, diarizer_output
            ... )
        """
        logger = logging.getLogger(f"{self.__class__.__name__}._merge_asr_with_diarization")
        logger.info(
            f"Fusionando {len(asr_segments)} segmentos ASR "
            f"con {len(diar_result.segments)} segmentos de diarización"
        )
        
        salida = []

        for seg in asr_segments:
            start = float(seg.get("start", 0.0))
            end = float(seg.get("end", 0.0))
            text = str(seg.get("text", "")).strip()

            if not text:
                logger.debug(f"Saltando segmento vacío [{format_min_sec(start)} - {format_min_sec(end)}]")
                continue

            speaker = self._speaker_by_max_overlap(
                start,
                end,
                diar_result.segments,
            )

            linea = (
                f"[{format_min_sec(start)} - {format_min_sec(end)}] "
                f"{speaker}: {text}"
            )
            salida.append(linea)
            logger.debug(f"Añadido segmento: {speaker} en [{format_min_sec(start)} - {format_min_sec(end)}]")

        logger.info(f"Generados {len(salida)} segmentos con diarización")
        return "\n".join(salida)

    @staticmethod
    def _speaker_by_max_overlap(start, end, diar_segments):
        """
        Determina el hablante más probable basado en máximo solapamiento temporal.
        
        Calcula el solapamiento temporal entre un segmento de texto [start, end]
        y todos los segmentos de diarización. El hablante del segmento con mayor
        solapamiento se asigna como el hablante del texto.
        
        Fórmula de solapamiento:
        overlap = max(0, min(end, seg_end) - max(start, seg_start))
        
        Args:
            start (float): Tiempo de inicio en segundos.
            end (float): Tiempo de finalización en segundos.
            diar_segments: Lista de segmentos de diarización con atributos
                start, end, speaker.
        
        Returns:
            str: Nombre/ID del hablante con máximo solapamiento.
                Devuelve "unknown" si no hay solapamiento.
        
        Ejemplo:
            >>> speaker = SpeechTranscriber._speaker_by_max_overlap(
            ...     start=5.0, end=12.0, diar_segments=diar_segs
            ... )
            >>> print(speaker)
            'Speaker_0'
        """
        logger = logging.getLogger(f"{SpeechTranscriber.__name__}._speaker_by_max_overlap")
        logger.debug(f"Determinando hablante para intervalo [{start} - {end}]")
        
        best_speaker = "unknown"
        best_overlap = 0.0

        for dseg in diar_segments:
            d_start = float(dseg.start)
            d_end = float(dseg.end)
            d_speaker = dseg.speaker

            overlap = max(
                0.0,
                min(end, d_end) - max(start, d_start),
            )

            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = d_speaker
                logger.debug(
                    f"Nuevo mejor match: {d_speaker} con solapamiento {overlap:.3f}s"
                )

        logger.debug(f"Hablante seleccionado: {best_speaker} (solapamiento: {best_overlap:.3f}s)")
        return best_speaker


In [ ]:
#export

class AudioTranscription:
    ENGINE_WHISPER = SpeechTranscriber.ENGINE_WHISPER
    ENGINE_MLX_AUDIO = SpeechTranscriber.ENGINE_MLX_AUDIO

    CLEAN_NONE = AudioCleaner.CLEAN_NONE
    CLEAN_FFMPEG = AudioCleaner.CLEAN_FFMPEG
    CLEAN_DEEPFILTER = AudioCleaner.CLEAN_DEEPFILTER

    def __init__(
        self,
        engine=ENGINE_WHISPER,
        diarization=True,
        clean_method=CLEAN_NONE,
        split=False,
        segment_minutes=25,
        output_dir=None,
        whisper_model="mlx-community/whisper-large-v3-mlx",
        mlx_audio_asr_model="mlx-community/parakeet-tdt-0.6b-v2",
        diar_model="mlx-community/diar_sortformer_4spk-v1-fp32",
        diar_threshold=0.5,
    ):
        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.debug("AudioTranscription inicializado")
        self.engine = engine
        self.diarization = diarization
        self.clean_method = clean_method
        self.split = split
        self.segment_minutes = segment_minutes
        self.output_dir = Path(output_dir) if output_dir else None

        self.cleaner = AudioCleaner()

        self.transcriber = SpeechTranscriber(
            engine=engine,
            diarization=diarization,
            whisper_model=whisper_model,
            mlx_audio_asr_model=mlx_audio_asr_model,
            diar_model=diar_model,
            diar_threshold=diar_threshold,
        )

    def process(self, input_file):
        """
        Ejecuta la pipeline completa de procesamiento para un archivo de audio.
        
        El proceso incluye:
        1. División del audio en segmentos (si `split=True`).
        2. Limpieza de cada segmento (si `clean_method` no es `CLEAN_NONE`).
        3. Transcripción de cada segmento usando el motor configurado.
        4. Generación de archivos `.txt` individuales para cada segmento.
        5. Combinación de todos los segmentos en un archivo de transcripción final.
        
        Args:
            input_file (str or Path): Ruta del archivo de audio a procesar.
            
        Returns:
            str: Ruta del archivo de transcripción final generado.
            
        Raises:
            RuntimeError: Si ocurre un error durante el proceso de división o procesamiento.
        
        Ejemplo:
            >>> pipeline = AudioTranscription(split=True, clean_method=AudioCleaner.CLEAN_FFMPEG)
            >>> final_txt = pipeline.process("audio.mp3")
        """
        self.logger.info(f"Iniciando proceso para: {input_file}")
        input_file = Path(input_file)

        output_dir = self.output_dir or input_file.parent
        output_dir.mkdir(parents=True, exist_ok=True)

        if self.split:
            self.logger.info(f"Dividiendo audio en segmentos de {self.segment_minutes} minutos")
            segmentos = self.divideFichero(
                input_file=input_file,
                minutos=self.segment_minutes,
                output_dir=output_dir,
            )
        else:
            duracion = self.get_audio_duration(input_file)
            self.logger.debug(f"Duración del audio: {duracion}s")
            segmentos = [
                {
                    "file": input_file,
                    "index": 1,
                    "start": 0.0,
                    "end": duracion,
                }
            ]

        txt_segmentos = []

        for segmento in segmentos:
            segment_file = Path(segmento["file"])
            index = segmento["index"]
            start = segmento["start"]
            end = segmento["end"]

            self.logger.info(f"Procesando segmento {index:02d}: {segment_file.name} [{format_min_sec(start)} - {format_min_sec(end)}]")
            print("\\n" + "=" * 80)
            print(f"Segmento {index:02d}")
            print(f"Archivo: {segment_file}")
            print(f"Rango original: {format_min_sec(start)} - {format_min_sec(end)}")
            print("=" * 80)

            audio_to_transcribe = segment_file

            if self.clean_method != self.CLEAN_NONE:
                clean_output = output_dir / f"{segment_file.stem}_limpia.wav"
                self.logger.info(format_min_sec(start))
                self.logger.info(f"Aplicando limpieza ({self.clean_method}) a: {segment_file.name}")
                audio_to_transcribe = Path(
                    self.cleaner.clean(
                        input_file=segment_file,
                        method=self.clean_method,
                        output_file=clean_output,
                    )
                )

            texto = self.transcriber.transcribe(audio_to_transcribe)

            txt_file = audio_to_transcribe.with_suffix(".txt")

            txt_content = (
                f"[segmento {index:02d} "
                f"{format_min_sec(start)} - {format_min_sec(end)}]\\n\\n"
                f"{texto.strip()}\\n"
            )

            txt_file.write_text(txt_content, encoding="utf-8")

            txt_segmentos.append(
                {
                    "txt": txt_file,
                    "index": index,
                    "start": start,
                    "end": end,
                }
            )

            self.logger.debug(f"TXT parcial generado: {txt_file}")
            print(f"TXT parcial generado: {txt_file}")

        txt_final = output_dir / f"{input_file.stem}_transcripcion_final.txt"

        self.logger.info(f"Generando archivo de transcripción final: {txt_final}")
        self._write_final_txt(
            txt_segmentos=txt_segmentos,
            output_file=txt_final,
        )

        self.logger.info(f"Proceso completado exitosamente. Resultado: {txt_final}")
        print("\\n" + "=" * 80)
        print(f"TXT final generado: {txt_final}")
        print("=" * 80)

        return str(txt_final)

    def divideFichero(self, input_file, minutos=25, output_dir=None):
        """
        Divide un archivo de audio en segmentos de duración configurable.
        
        Utiliza FFmpeg para segmentar el archivo de audio en partes de la duración
        especificada, manteniendo la calidad y el formato mono 16kHz.
        
        Args:
            input_file (str or Path): Ruta del archivo de audio original.
            minutos (int, optional): Duración de cada segmento en minutos.
                Por defecto: 25.
            output_dir (str or Path, optional): Directorio donde se guardarán los segmentos.
                Si es None, se usa el directorio del archivo de entrada.
                
        Returns:
            list: Lista de diccionarios, cada uno representando un segmento con:
                - 'file' (Path): Ruta al archivo de segmento.
                - 'index' (int): Número de segmento.
                - 'start' (float): Tiempo de inicio en segundos.
                - 'end' (float): Tiempo de fin en segundos.
                
        Raises:
            RuntimeError: Si no se puede obtener la duración del archivo o si FFmpeg falla.
        
        Ejemplo:
            >>> segments = pipeline.divideFichero("audio.mpint", minutos=10)
        """
        self.logger.info(f"Dividiendo archivo: {input_file} en segmentos de {minutos} min")
        input_file = Path(input_file)

        if output_dir is None:
            output_dir = input_file.parent
        else:
            output_dir = Path(output_dir)
            output_dir.mkdir(parents=True, exist_ok=True)

        total_duration = self.get_audio_duration(input_file)

        if total_duration <= 0:
            error_msg = f"No se pudo obtener duración de {input_file}"
            self.logger.error(error_msg)
            raise RuntimeError(error_msg)

        segment_duration = minutos * 60
        num_segments = math.ceil(total_duration / segment_duration)
        self.logger.info(f"Total de segmentos a generar: {num_segments}")

        segmentos = []

        for i in range(num_segments):
            start = i * segment_duration
            end = min(start + segment_duration, total_duration)
            duration = end - start

            output_segment = (
                output_dir
                / f"{input_file.stem}_segmento_{i + 1:02d}"
                f"_{format_min_sec_filename(start)}"
                f"_{format_min_sec_filename(end)}.wav"
            )

            comando = [
                "ffmpeg",
                "-i", str(input_file),
                "-ss", str(start),
                "-t", str(duration),
                "-vn",
                "-ac", "1",
                "-ar", "16000",
                "-c:a", "pcm_s16le",
                "-y",
                str(output_segment),
            ]

            self.logger.debug(f"Ejecutando segmentación: {' '.join(comando)}")
            AudioCleaner._run_command(
                comando,
                f"ERROR dividiendo segmento {i + 1}:",
            )

            segmentos.append(
                {
                    "file": output_segment,
                    "format_min_sec_filename": format_min_sec_filename(start), # Just for context
                    "index": i + 1,
                    "start": start,
                    "end": end,
                }
            )
            self.logger.debug(f"Segmento {i+1} creado: {output_segment}")

        return segmentos

    def _write_final_txt(self, txt_segmentos, output_file):
        """
        Combina todos los archivos de texto de los segmentos en un único archivo final.
        
        Args:
            txt_segmentos (list): Lista de diccionarios con la información de cada segmento.
                Cada dict debe contener 'txt' (Path al archivo .txt).
            output_file (str or Path): Ruta del archivo de texto final a generar.
            
        Returns:
            str: Ruta del archivo final generado.
        """
        self.logger.info(f"Combinando {len(txt_segmentos)} segmentos en: {output_file}")
        output_file = Path(output_file)

        partes = []

        for item in sorted(txt_segmentos, key=lambda x: x["index"]):
            txt_file = Path(item["txt"])
            index = item["index"]
            start = item["start"]
            end = item["end"]

            if not txt_file.exists():
                self.logger.warning(f"Archivo de segmento no encontrado: {txt_file}")
                continue

            contenido = txt_file.read_text(encoding="utf-8").strip()

            bloque = (
                "\\n"
                + "=" * 80
                + "\\n"
                + f"[segmento {index:02d} "
                + f"{format_min_sec(start)} - {format_min_sec(end)}]"
                + "\\n"
                + "=" * 80
                + "\\n\\n"
                + contenido
                + "\\n"
            )

            partes.append(bloque)

        output_file.write_text(
            "\\n".join(partes),
            encoding="utf-8",
        )

        self.logger.info(f"Archivo final escrito exitosamente: {output_file}")
        return str(output_file)

    @staticmethod
    def get_audio_duration(input_file):
        """
        Obtiene la duración de un archivo de audio usando ffprobe.
        
        Args:
            input_file (str or Path): Ruta del archivo de audio.
            
        Returns:
            float: Duración del audio en segundos.
            
        Raises:
            RuntimeError: Si ffprobe falla o no puede obtener la duración.
        
        Ejemplo:
            >>> duration = AudioTranscription.get_audio_duration("audio.mp3")
            >>> print(duration)
            120.5
        """
        input_file = Path(input_file)

        comando = [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_flags=1:nokey=1",
            str(input_file),
        ]

        resultado = subprocess.run(
            comando,
            capture_output=True,
            text=True,
        )

        if resultado.returncode != 0:
            error_msg = (
                f"ERROR obteniendo duración de {input_file}:\\n{resultado.stderr}"
            )
            raise RuntimeError(error_msg)

        return float(resultado.stdout.strip())

    def process_directory(self, input_dir):
        """
        Procesa todos los archivos de audio en un directorio.

        Busca archivos con extensiones de audio comunes, descarta los
        segmentos generados por la clase (_segmento_) y omite los
        que ya tienen su archivo .txt de transcripción existente.
        """
        self.logger.info(f"Iniciando procesamiento de directorio: {input_dir}")
        input_dir = Path(input_dir)
        if not input_dir.is_dir():
            error_msg = f"El directorio no existe: {input_dir}"
            self.logger.error(error_msg)
            raise ValueError(error_msg)

        audio_extensions = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}

        audio_files = [
            f
            for f in input_dir.iterdir()
            if f.is_file()
            and f.suffix.lower() in audio_extensions
            and "_segmento_" not in f.name
        ]

        if not audio_files:
            self.logger.warning(f"No se encontraron archivos de audio en {input_dir}")
            print(f"No se encontraron archivos de audio en {input_dir}")
            return

        self.logger.info(f"Encontrados {len(audio_files)} archivos de audio para procesar.")
        print(f"Encontrados {len(audio_files)} archivos de audio para proces. ")

        for audio_file in audio_files:
            output_dir = self.output_dir or audio_file.parent
            txt_final = Path(output_dir) / f"{audio_file.stem}_transcripcion_final.txt"

            if txt_final.exists():
                self.logger.info(f"Saltando: {audio_file.name} (ya procesado: {txt_final.name})")
                print(f"Saltando: {audio_file.name} (ya procesado: {txt_final.name})")
                continue

            self.logger.info(f"Procesando: {audio_file.name}")
            print(f"Procesando: {audio_file.name}")
            try:
                self.process(audio_file)
            except Exception as e:
                error_msg = f"Error procesando {audio_file.name}: {e}"
                self.logger.error(error_msg)
                print(error_msg)


def format_min_sec(seconds):
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes:02d}:{secs:02d}"


def format_min_sec_filename(seconds):
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes:02d}m{secs:02d}s"


In [ ]:

def process_directory_gemma(self, input_dir):
    input_dir = Path(input_dir)
    if not input_dir.is_dir():
        raise ValueError(f"El directorio no existe: {input_dir}")

    audio_extensions = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}
        
    audio_files = [
            f for
            f in input_dir.iterdir()
            if f.is_file() and f.suffix.lower() in audio_extensions
            and "_segmento_" not in f.name
    ]

    if not audio_files:
        print(f"No se encontraron archivos de audio en {input_dir}")
        return

    print(f"Encontrados {len(audio_files)} archivos de audio para procesar.")

    for audio_file in audio_files:
        output_dir = self.output_dir or audio_file.parent
        txt_final = Path(output_dir) / f"{audio_file.stem}_transcripcion_final.txt"

        if txt_final.exists():
            print(f"Saltando: {audio_file.name} (ya procesado: {txt_final.name})")
            continue

        print(f"Procesando: {audio_file.name}")
        try:
            self.process(audio_file)
        except Exception as e:
            print(f"Error procesando {audio_file.name}: {e}")
